# External Lab Evaluation V3 Corrected: Closed-Set RefMet Normalization

V3 evaluates the task you can actually support with the data on hand:

```text
query:  metabolite_name
answer: refmet_name from the lab-provided RefMet label set
```

This is a closed-set normalization experiment. It does **not** assume access to an external official RefMet vocabulary. Instead, it encodes the unique `refmet_name` values present in the lab file, searches that label set with `metabolite_name`, tunes a lightweight reranker on validation rows, and reports held-out test performance.

V3 answers:

1. Can SapBERT retrieve the correct lab-provided RefMet label directly?
2. Does a tuned hybrid reranker improve over pure embedding similarity?
3. Can confidence thresholds identify a high-precision subset for automated normalization?


## Install Dependencies


In [ ]:
import subprocess
import sys

base_packages = [
    'pandas==2.2.2',
    'numpy>=1.26.4,<2.1',
    'pyarrow>=15.0',
    'tqdm>=4.66',
    'torch>=2.1',
    'transformers>=4.40',
    'accelerate>=0.28',
    'sentencepiece>=0.2',
    'matplotlib>=3.8',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *base_packages])

try:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'faiss-gpu-cu12'])
except subprocess.CalledProcessError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'faiss-cpu>=1.8'])

print('Installed notebook dependencies.')


## Mount Drive and Locate Repo


In [ ]:
from pathlib import Path
import gc
import json
import os
import re
import shutil
import sys
import time
import unicodedata
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', exc)


def find_repo_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path('/content/drive/MyDrive/ChemSynonymizer/chem_synonymizer'),
        Path('/content/drive/MyDrive/chem_synonymizer'),
        Path('/content/chem_synonymizer'),
    ]
    for candidate in candidates:
        if (candidate / 'src' / 'chem_synonymizer').exists():
            return candidate.resolve()
    raise FileNotFoundError('Could not find repo root containing src/chem_synonymizer.')


REPO_DIR = find_repo_dir()
sys.path.insert(0, str(REPO_DIR / 'src'))
print('Repo:', REPO_DIR)


## Configuration


In [ ]:
from chem_synonymizer.embed import load_hf_model, pool_outputs
from chem_synonymizer.utils import save_json

LAB_CSV_PATH = REPO_DIR / 'data' / 'allmetnames_062626.csv'
MODEL_DIR = REPO_DIR / 'results' / 'sapbert_contrastive' / 'artifacts' / 'model'

METABOLITE_COL = 'metabolite_name'
REFMET_COL = 'refmet_name'

RUN_NAME = 'sapbert_contrastive_refmet_v3_closed_set'
RESULTS_RUN_DIR = REPO_DIR / 'results' / 'sapbert_contrastive' / 'external_lab_v3_corrected_closed_set'
RESULTS_ARTIFACTS_DIR = RESULTS_RUN_DIR / 'artifacts'

SCRATCH_ROOT = Path('/mnt/local-scratch') if Path('/mnt/local-scratch').exists() else Path('/content')
WORK_DIR = SCRATCH_ROOT / 'chem_synonymizer' / RUN_NAME
WORK_ARTIFACTS_DIR = WORK_DIR / 'artifacts'

QUERY_EMBEDDINGS_PATH = WORK_ARTIFACTS_DIR / 'metabolite_query_embeddings.npy'
REFMET_EMBEDDINGS_PATH = WORK_ARTIFACTS_DIR / 'refmet_label_embeddings.npy'
NEIGHBOR_INDICES_PATH = WORK_ARTIFACTS_DIR / 'closed_refmet_neighbor_indices.npy'
NEIGHBOR_SCORES_PATH = WORK_ARTIFACTS_DIR / 'closed_refmet_neighbor_scores.npy'

SEED = 7
TRAIN_FRAC = 0.60
VAL_FRAC = 0.20
TEST_FRAC = 0.20

MAX_LENGTH = 96
EMBED_DTYPE = 'float16'
POOLING = 'cls'
L2_NORMALIZE = True
QUERY_ENCODE_BATCH_SIZE = 8192
REFMET_ENCODE_BATCH_SIZE = 8192
SEARCH_BATCH_SIZE = 8192
K_RETRIEVE = 256

# Reranker tuning/evaluation settings. Increase these for slower but broader reranking.
RERANK_POOL_K = 64
TUNE_VAL_SAMPLE_SIZE = 15000
TUNE_RANDOM_WEIGHT_TRIALS = 120
RERANK_BATCH_SIZE = 2000
SAVE_TOP_K = 10

REENCODE_QUERIES = False
REENCODE_REFMET_LABELS = False
RERUN_SEARCH = False
RERUN_TUNING = True

for required in [LAB_CSV_PATH, MODEL_DIR]:
    if not required.exists():
        raise FileNotFoundError(required)
for path in [RESULTS_RUN_DIR, RESULTS_ARTIFACTS_DIR, WORK_ARTIFACTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('Lab CSV:', LAB_CSV_PATH)
print('Model:', MODEL_DIR)
print('Results:', RESULTS_RUN_DIR)
print('Scratch:', WORK_DIR, shutil.disk_usage(SCRATCH_ROOT))


## Helpers


In [ ]:
GREEK_MAP = {
    'α': 'alpha', 'β': 'beta', 'γ': 'gamma', 'δ': 'delta',
    'ε': 'epsilon', 'κ': 'kappa', 'λ': 'lambda', 'μ': 'mu',
    'ω': 'omega',
}


def normalize_text(value: object) -> str:
    if pd.isna(value):
        return ''
    text = unicodedata.normalize('NFKC', str(value))
    for src, dst in GREEK_MAP.items():
        text = text.replace(src, dst)
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)
    return text


def normalize_series(values: pd.Series) -> pd.Series:
    return values.fillna('').astype('string').map(normalize_text).astype('string')


def compact_key(text: object) -> str:
    return re.sub(r'[^a-z0-9]+', '', normalize_text(text))


def char_ngrams(text: str, n: int = 3) -> set[str]:
    s = compact_key(text)
    if not s:
        return set()
    if len(s) <= n:
        return {s}
    return {s[i:i+n] for i in range(len(s) - n + 1)}


def token_set(text: str) -> set[str]:
    return {tok for tok in re.split(r'[^a-z0-9]+', normalize_text(text)) if tok}


def jaccard(a: set[str], b: set[str]) -> float:
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return len(a & b) / max(1, len(a | b))


def sequence_ratio(a: str, b: str) -> float:
    a = normalize_text(a)
    b = normalize_text(b)
    if len(a) > 120 or len(b) > 120:
        return jaccard(char_ngrams(a, 3), char_ngrams(b, 3))
    return SequenceMatcher(None, a, b).ratio()


def lipid_signature(text: str) -> tuple[str, str, str] | None:
    s = normalize_text(text)
    m = re.search(r'\b([a-z]{1,6})\s*[\( ]?\s*(\d{1,2})\s*:\s*(\d{1,2})\b', s)
    if not m:
        return None
    return (m.group(1), m.group(2), m.group(3))


def stereo_signature(text: str) -> tuple[str, ...]:
    s = normalize_text(text)
    toks = re.findall(r'\b(?:cis|trans|alpha|beta)\b|\b[dl]-|\([+-]\)|\b[0-9]+[rs]\b|\b[0-9]+[ez]\b', s)
    return tuple(sorted(toks))


def text_profile(text: str) -> dict:
    return {
        'norm': normalize_text(text),
        'compact': compact_key(text),
        'char3': char_ngrams(text, 3),
        'tokens': token_set(text),
        'lipid': lipid_signature(text),
        'stereo': stereo_signature(text),
    }


def lexical_feature_dict(query_profile: dict, candidate_profile: dict) -> dict:
    q = query_profile
    c = candidate_profile
    return {
        'compact_exact': float(q['compact'] == c['compact'] and q['compact'] != ''),
        'sequence_ratio': float(sequence_ratio(q['norm'], c['norm'])),
        'char3_jaccard': float(jaccard(q['char3'], c['char3'])),
        'token_jaccard': float(jaccard(q['tokens'], c['tokens'])),
        'lipid_signature_match': float(q['lipid'] is not None and q['lipid'] == c['lipid']),
        'lipid_class_match': float(q['lipid'] is not None and c['lipid'] is not None and q['lipid'][0] == c['lipid'][0]),
        'stereo_mismatch': float(bool(q['stereo'] and c['stereo'] and q['stereo'] != c['stereo'])),
    }


def save_config() -> None:
    save_json(
        {
            'run_name': RUN_NAME,
            'task': 'closed-set lab-provided refmet_name normalization from metabolite_name',
            'lab_csv': str(LAB_CSV_PATH),
            'model_dir': str(MODEL_DIR),
            'split': {'seed': SEED, 'train_frac': TRAIN_FRAC, 'val_frac': VAL_FRAC, 'test_frac': TEST_FRAC},
            'max_length': MAX_LENGTH,
            'pooling': POOLING,
            'embedding_dtype': EMBED_DTYPE,
            'k_retrieve': K_RETRIEVE,
            'rerank_pool_k': RERANK_POOL_K,
            'tune_val_sample_size': TUNE_VAL_SAMPLE_SIZE,
            'tune_random_weight_trials': TUNE_RANDOM_WEIGHT_TRIALS,
        },
        RESULTS_RUN_DIR / 'config.json',
    )


def load_json_if_exists(path: Path) -> dict | None:
    if not path.exists():
        return None
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)


save_config()


## Load Lab Pairs and Build Closed RefMet Label Set


In [ ]:
def load_lab_pairs_and_targets() -> tuple[pd.DataFrame, pd.DataFrame]:
    lab = pd.read_csv(LAB_CSV_PATH, dtype='string', engine='python', on_bad_lines='skip')
    missing = [col for col in [METABOLITE_COL, REFMET_COL] if col not in lab.columns]
    if missing:
        raise ValueError(f'Missing required columns: {missing}. Available columns: {list(lab.columns)}')

    lab = lab.reset_index(drop=True).copy()
    lab['source_row_id'] = np.arange(len(lab), dtype=np.int64)
    lab['metabolite_name'] = lab[METABOLITE_COL].astype('string').str.strip()
    lab['refmet_name'] = lab[REFMET_COL].astype('string').str.strip()
    lab['metabolite_normalized'] = normalize_series(lab['metabolite_name'])
    lab['refmet_normalized'] = normalize_series(lab['refmet_name'])
    keep = (lab['metabolite_normalized'].str.len() > 0) & (lab['refmet_normalized'].str.len() > 0)
    pairs = lab[keep].copy()
    pairs = pairs.drop_duplicates(['metabolite_normalized', 'refmet_normalized']).reset_index(drop=True)

    targets = (
        pairs.groupby('refmet_normalized', sort=True)
        .agg(refmet_name=('refmet_name', 'first'), n_query_pairs=('source_row_id', 'size'))
        .reset_index()
    )
    targets['target_id'] = np.arange(len(targets), dtype=np.int64)
    target_lookup = targets.set_index('refmet_normalized')['target_id'].to_dict()
    pairs['target_id'] = pairs['refmet_normalized'].map(target_lookup).astype(np.int64)
    pairs['query_id'] = np.arange(len(pairs), dtype=np.int64)

    rng = np.random.default_rng(SEED)
    order = rng.permutation(len(pairs))
    split = np.empty(len(pairs), dtype=object)
    n_train = int(round(TRAIN_FRAC * len(pairs)))
    n_val = int(round(VAL_FRAC * len(pairs)))
    split[order[:n_train]] = 'train'
    split[order[n_train:n_train+n_val]] = 'val'
    split[order[n_train+n_val:]] = 'test'
    pairs['split'] = split

    pairs.to_csv(RESULTS_ARTIFACTS_DIR / 'v3_closed_set_query_pairs.csv', index=False)
    targets.to_csv(RESULTS_ARTIFACTS_DIR / 'v3_refmet_label_dictionary.csv', index=False)
    summary = {
        'raw_lab_rows': int(len(lab)),
        'rows_with_both_names': int(keep.sum()),
        'deduplicated_query_pairs': int(len(pairs)),
        'unique_refmet_labels': int(len(targets)),
        'split_counts': pairs['split'].value_counts().to_dict(),
        'identical_metabolite_refmet_pairs': int(pairs['metabolite_normalized'].eq(pairs['refmet_normalized']).sum()),
    }
    save_json(summary, RESULTS_RUN_DIR / 'v3_closed_set_data_summary.json')
    print(json.dumps(summary, indent=2))
    return pairs, targets


lab_pairs, refmet_targets = load_lab_pairs_and_targets()
display(lab_pairs.head(10))
display(refmet_targets.head(10))


## Encode Queries and RefMet Labels


In [ ]:
import torch


def encode_texts_to_memmap(
    texts: list[str],
    out_path: Path,
    tokenizer,
    model,
    model_name: str,
    batch_size: int,
    device: str,
    run_key: str,
    force_reencode: bool = False,
) -> np.ndarray:
    n = len(texts)
    d = int(model.config.hidden_size)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    meta_path = out_path.with_suffix(out_path.suffix + '.meta.json')
    expected_meta = {
        'n': n,
        'd': d,
        'model_name': model_name,
        'pooling': POOLING,
        'max_length': int(MAX_LENGTH),
        'dtype': EMBED_DTYPE,
        'normalize': bool(L2_NORMALIZE),
        'run_key': run_key,
    }
    if out_path.exists() and meta_path.exists() and not force_reencode:
        old_meta = load_json_if_exists(meta_path)
        arr = np.load(out_path, mmap_mode='r')
        if old_meta == expected_meta and arr.shape == (n, d):
            print('Using cached embeddings:', out_path)
            return arr
    if out_path.exists():
        out_path.unlink()
    arr = np.lib.format.open_memmap(
        out_path,
        mode='w+',
        dtype=np.float16 if EMBED_DTYPE == 'float16' else np.float32,
        shape=(n, d),
    )
    autocast_enabled = device.startswith('cuda') and torch.cuda.is_available()
    t0 = time.perf_counter()
    with torch.inference_mode():
        for start in tqdm(range(0, n, int(batch_size)), desc=f'Encoding {out_path.name}'):
            end = min(start + int(batch_size), n)
            batch = [str(x) for x in texts[start:end]]
            enc = tokenizer(batch, padding=True, truncation=True, max_length=int(MAX_LENGTH), return_tensors='pt').to(device)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=autocast_enabled):
                out = model(**enc)
                emb = pool_outputs(out.last_hidden_state, enc['attention_mask'], POOLING)
                if L2_NORMALIZE:
                    emb = torch.nn.functional.normalize(emb, p=2, dim=1)
            arr[start:end] = emb.detach().cpu().numpy().astype(arr.dtype, copy=False)
            del batch, enc, out, emb
            if device.startswith('cuda'):
                torch.cuda.empty_cache()
    arr.flush()
    save_json(expected_meta, meta_path)
    save_json({'elapsed_sec': float(time.perf_counter() - t0)}, out_path.with_suffix(out_path.suffix + '.timing.json'))
    return np.load(out_path, mmap_mode='r')


device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Embedding device:', device)
tokenizer, hf_model = load_hf_model(str(MODEL_DIR), device)

query_embeddings = encode_texts_to_memmap(
    lab_pairs['metabolite_normalized'].astype(str).tolist(),
    QUERY_EMBEDDINGS_PATH,
    tokenizer,
    hf_model,
    str(MODEL_DIR),
    QUERY_ENCODE_BATCH_SIZE,
    device,
    run_key=RUN_NAME + '_queries',
    force_reencode=REENCODE_QUERIES,
)

refmet_embeddings = encode_texts_to_memmap(
    refmet_targets['refmet_normalized'].astype(str).tolist(),
    REFMET_EMBEDDINGS_PATH,
    tokenizer,
    hf_model,
    str(MODEL_DIR),
    REFMET_ENCODE_BATCH_SIZE,
    device,
    run_key=RUN_NAME + '_refmet_labels',
    force_reencode=REENCODE_REFMET_LABELS,
)

del hf_model
del tokenizer
gc.collect()
if device.startswith('cuda'):
    torch.cuda.empty_cache()
print('Query embeddings:', query_embeddings.shape, query_embeddings.dtype)
print('RefMet label embeddings:', refmet_embeddings.shape, refmet_embeddings.dtype)


## Exact Closed-Set RefMet Search


In [ ]:
import faiss


def build_refmet_index(embeddings: np.ndarray, device: str):
    dim = embeddings.shape[1]
    cpu_index = faiss.IndexFlatIP(dim)
    resource = None
    index = cpu_index
    if device.startswith('cuda') and hasattr(faiss, 'StandardGpuResources'):
        resource = faiss.StandardGpuResources()
        opts = faiss.GpuClonerOptions()
        opts.useFloat16 = True
        index = faiss.index_cpu_to_gpu(resource, 0, cpu_index, opts)
    for start in tqdm(range(0, len(embeddings), 250000), desc='Adding RefMet labels to exact index'):
        end = min(start + 250000, len(embeddings))
        xb = np.ascontiguousarray(np.asarray(embeddings[start:end], dtype=np.float32))
        index.add(xb)
        del xb
    return index, resource


def search_index(index, query_embeddings: np.ndarray, k: int, batch_size: int) -> tuple[np.ndarray, np.ndarray]:
    n = len(query_embeddings)
    idx = np.lib.format.open_memmap(NEIGHBOR_INDICES_PATH, mode='w+', dtype=np.int64, shape=(n, int(k)))
    scores = np.lib.format.open_memmap(NEIGHBOR_SCORES_PATH, mode='w+', dtype=np.float32, shape=(n, int(k)))
    for start in tqdm(range(0, n, int(batch_size)), desc='Searching closed RefMet label set'):
        end = min(start + int(batch_size), n)
        xq = np.ascontiguousarray(np.asarray(query_embeddings[start:end], dtype=np.float32))
        d, i = index.search(xq, int(k))
        idx[start:end] = i
        scores[start:end] = d
        del xq, d, i
    idx.flush()
    scores.flush()
    return np.load(NEIGHBOR_INDICES_PATH, mmap_mode='r'), np.load(NEIGHBOR_SCORES_PATH, mmap_mode='r')


search_meta_path = NEIGHBOR_INDICES_PATH.with_suffix(NEIGHBOR_INDICES_PATH.suffix + '.meta.json')
expected_search_meta = {
    'n_queries': int(len(lab_pairs)),
    'n_targets': int(len(refmet_targets)),
    'k_retrieve': int(K_RETRIEVE),
    'query_shape': tuple(query_embeddings.shape),
    'target_shape': tuple(refmet_embeddings.shape),
    'run_name': RUN_NAME,
}

if (
    NEIGHBOR_INDICES_PATH.exists()
    and NEIGHBOR_SCORES_PATH.exists()
    and load_json_if_exists(search_meta_path) == expected_search_meta
    and not RERUN_SEARCH
):
    neighbor_indices = np.load(NEIGHBOR_INDICES_PATH, mmap_mode='r')
    neighbor_scores = np.load(NEIGHBOR_SCORES_PATH, mmap_mode='r')
    print('Using cached closed-set neighbors:', neighbor_indices.shape)
else:
    refmet_index, refmet_index_resource = build_refmet_index(refmet_embeddings, device)
    neighbor_indices, neighbor_scores = search_index(refmet_index, query_embeddings, K_RETRIEVE, SEARCH_BATCH_SIZE)
    save_json(expected_search_meta, search_meta_path)
print('Closed-set neighbors:', neighbor_indices.shape)


## Candidate Metrics and Reranker Feature Builder


In [ ]:
target_norms = refmet_targets['refmet_normalized'].astype(str).tolist()
target_display = refmet_targets['refmet_name'].astype(str).tolist()
target_profiles = [text_profile(x) for x in tqdm(target_norms, desc='Precomputing target text profiles')]
query_norms = lab_pairs['metabolite_normalized'].astype(str).tolist()
query_profiles = [text_profile(x) for x in tqdm(query_norms, desc='Precomputing query text profiles')]
gold_target_ids = lab_pairs['target_id'].to_numpy(dtype=np.int64)


def ranks_from_neighbor_ids(row_indices: np.ndarray, k: int) -> np.ndarray:
    ranks = np.full(len(row_indices), np.inf, dtype=np.float64)
    for out_i, row_i in enumerate(row_indices):
        gold = int(gold_target_ids[row_i])
        hits = np.where(neighbor_indices[row_i, :k] == gold)[0]
        if len(hits):
            ranks[out_i] = int(hits[0] + 1)
    return ranks


def metrics_from_ranks(rank_arr: np.ndarray, prefix: str = '') -> dict:
    rank_arr = np.asarray(rank_arr, dtype=np.float64)
    finite = np.isfinite(rank_arr)
    n = len(rank_arr)
    out = {f'{prefix}MRR': float(np.sum(np.where(finite, 1.0 / rank_arr, 0.0)) / max(1, n))}
    for k in [1, 5, 10, 25, 50, 100, 256]:
        out[f'{prefix}HitRate@{k}'] = float(np.mean(rank_arr <= k)) if n else 0.0
    return out


FEATURE_NAMES = [
    'embedding_score_norm',
    'rank_prior',
    'compact_exact',
    'sequence_ratio',
    'char3_jaccard',
    'token_jaccard',
    'lipid_signature_match',
    'lipid_class_match',
    'stereo_mismatch',
]


def build_feature_block(row_indices: np.ndarray, pool_k: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    row_indices = np.asarray(row_indices, dtype=np.int64)
    cand_ids = np.asarray(neighbor_indices[row_indices, :pool_k], dtype=np.int64)
    emb_scores = np.asarray(neighbor_scores[row_indices, :pool_k], dtype=np.float32)
    n, k = cand_ids.shape
    features = np.zeros((n, k, len(FEATURE_NAMES)), dtype=np.float32)
    features[:, :, 0] = (emb_scores + 1.0) / 2.0
    features[:, :, 1] = 1.0 / (np.arange(k, dtype=np.float32)[None, :] + 1.0)

    for local_i, row_i in enumerate(tqdm(row_indices, desc='Building lexical feature block', leave=False)):
        qprof = query_profiles[int(row_i)]
        for j, target_id in enumerate(cand_ids[local_i]):
            if target_id < 0:
                continue
            feats = lexical_feature_dict(qprof, target_profiles[int(target_id)])
            features[local_i, j, 2] = feats['compact_exact']
            features[local_i, j, 3] = feats['sequence_ratio']
            features[local_i, j, 4] = feats['char3_jaccard']
            features[local_i, j, 5] = feats['token_jaccard']
            features[local_i, j, 6] = feats['lipid_signature_match']
            features[local_i, j, 7] = feats['lipid_class_match']
            features[local_i, j, 8] = feats['stereo_mismatch']
    return cand_ids, emb_scores, features


def ranks_from_scores(cand_ids: np.ndarray, scores: np.ndarray, row_indices: np.ndarray) -> np.ndarray:
    ranks = np.full(len(row_indices), np.inf, dtype=np.float64)
    order = np.argsort(-scores, axis=1)
    for local_i, row_i in enumerate(row_indices):
        gold = int(gold_target_ids[int(row_i)])
        ordered_ids = cand_ids[local_i, order[local_i]]
        hits = np.where(ordered_ids == gold)[0]
        if len(hits):
            ranks[local_i] = int(hits[0] + 1)
    return ranks


def score_features(features: np.ndarray, weights: dict) -> np.ndarray:
    w = np.array([weights.get(name, 0.0) for name in FEATURE_NAMES], dtype=np.float32)
    return np.tensordot(features, w, axes=([2], [0]))


## Baseline Closed-Set Metrics


In [ ]:
split_indices = {
    split: lab_pairs.index[lab_pairs['split'].eq(split)].to_numpy(dtype=np.int64)
    for split in ['train', 'val', 'test']
}

baseline_metrics = {}
for split, indices in split_indices.items():
    ranks = ranks_from_neighbor_ids(indices, K_RETRIEVE)
    baseline_metrics[split] = metrics_from_ranks(ranks, prefix='embedding_')
print(json.dumps(baseline_metrics, indent=2))
save_json(baseline_metrics, RESULTS_RUN_DIR / 'v3_embedding_baseline_metrics.json')


## Tune Hybrid Reranker Weights on Validation Split


In [ ]:
def evaluate_weight_vector(cand_ids: np.ndarray, features: np.ndarray, row_indices: np.ndarray, weights: dict) -> dict:
    scores = score_features(features, weights)
    ranks = ranks_from_scores(cand_ids, scores, row_indices)
    return metrics_from_ranks(ranks)


def make_weight_trials(rng: np.random.Generator) -> list[dict]:
    fixed = [
        {'name': 'embedding_only', 'embedding_score_norm': 1.0},
        {
            'name': 'v2_like',
            'embedding_score_norm': 0.72,
            'rank_prior': 0.04,
            'compact_exact': 0.06,
            'sequence_ratio': 0.06,
            'char3_jaccard': 0.05,
            'token_jaccard': 0.03,
            'lipid_signature_match': 0.04,
            'lipid_class_match': 0.02,
            'stereo_mismatch': -0.04,
        },
        {
            'name': 'lexical_heavy',
            'embedding_score_norm': 0.45,
            'rank_prior': 0.02,
            'compact_exact': 0.16,
            'sequence_ratio': 0.12,
            'char3_jaccard': 0.09,
            'token_jaccard': 0.08,
            'lipid_signature_match': 0.06,
            'lipid_class_match': 0.03,
            'stereo_mismatch': -0.06,
        },
        {
            'name': 'lipid_friendly',
            'embedding_score_norm': 0.55,
            'rank_prior': 0.03,
            'compact_exact': 0.08,
            'sequence_ratio': 0.06,
            'char3_jaccard': 0.06,
            'token_jaccard': 0.04,
            'lipid_signature_match': 0.14,
            'lipid_class_match': 0.06,
            'stereo_mismatch': -0.05,
        },
    ]
    names = [
        'embedding_score_norm',
        'rank_prior',
        'compact_exact',
        'sequence_ratio',
        'char3_jaccard',
        'token_jaccard',
        'lipid_signature_match',
        'lipid_class_match',
    ]
    trials = fixed.copy()
    alpha = np.array([8.0, 0.7, 1.0, 1.2, 1.0, 0.8, 0.8, 0.5])
    for i in range(int(TUNE_RANDOM_WEIGHT_TRIALS)):
        vals = rng.dirichlet(alpha)
        trial = {'name': f'random_{i:03d}'}
        trial.update({name: float(value) for name, value in zip(names, vals)})
        trial['stereo_mismatch'] = -float(rng.choice([0.0, 0.02, 0.04, 0.06, 0.08]))
        trials.append(trial)
    return trials


rng = np.random.default_rng(SEED)
val_indices = split_indices['val']
if len(val_indices) > int(TUNE_VAL_SAMPLE_SIZE):
    tune_indices = np.sort(rng.choice(val_indices, size=int(TUNE_VAL_SAMPLE_SIZE), replace=False))
else:
    tune_indices = val_indices
print('Tuning on validation rows:', len(tune_indices))

tuning_path = RESULTS_RUN_DIR / 'v3_reranker_tuning_results.csv'
weights_path = RESULTS_RUN_DIR / 'v3_best_reranker_weights.json'
if tuning_path.exists() and weights_path.exists() and not RERUN_TUNING:
    tuning_results = pd.read_csv(tuning_path)
    best_weights = load_json_if_exists(weights_path)
    print('Loaded cached tuning results.')
else:
    tune_cand_ids, tune_emb_scores, tune_features = build_feature_block(tune_indices, RERANK_POOL_K)
    rows = []
    for trial in tqdm(make_weight_trials(rng), desc='Tuning reranker weights'):
        metrics = evaluate_weight_vector(tune_cand_ids, tune_features, tune_indices, trial)
        row = {'name': trial['name'], **{k: trial.get(k, 0.0) for k in FEATURE_NAMES}, **metrics}
        rows.append(row)
    tuning_results = pd.DataFrame(rows).sort_values(['MRR', 'HitRate@1', 'HitRate@10'], ascending=False).reset_index(drop=True)
    best_weights = tuning_results.iloc[0][['name', *FEATURE_NAMES]].to_dict()
    tuning_results.to_csv(tuning_path, index=False)
    save_json(best_weights, weights_path)
display(tuning_results.head(15))
print('Best weights:', json.dumps(best_weights, indent=2))


## Evaluate Tuned Reranker on Train/Val/Test


In [ ]:
def evaluate_reranker_split(row_indices: np.ndarray, split_name: str, save_predictions: bool = False):
    row_indices = np.asarray(row_indices, dtype=np.int64)
    all_ranks = []
    prediction_rows = []
    top_rows = []

    weight_dict = {name: float(best_weights.get(name, 0.0)) for name in FEATURE_NAMES}
    for start in tqdm(range(0, len(row_indices), int(RERANK_BATCH_SIZE)), desc=f'Evaluating reranker {split_name}'):
        batch_indices = row_indices[start:start+int(RERANK_BATCH_SIZE)]
        cand_ids, emb_scores, features = build_feature_block(batch_indices, RERANK_POOL_K)
        scores = score_features(features, weight_dict)
        order = np.argsort(-scores, axis=1)
        ranks = ranks_from_scores(cand_ids, scores, batch_indices)
        all_ranks.append(ranks)

        if save_predictions:
            for local_i, row_i in enumerate(batch_indices):
                qrow = lab_pairs.iloc[int(row_i)]
                ordered = order[local_i]
                top_j = int(ordered[0])
                top_target_id = int(cand_ids[local_i, top_j])
                gold = int(qrow['target_id'])
                top2_score = float(scores[local_i, ordered[1]]) if len(ordered) > 1 else float('nan')
                top_score = float(scores[local_i, top_j])
                best_rank = ranks[local_i]
                prediction_rows.append({
                    'query_id': int(qrow['query_id']),
                    'source_row_id': int(qrow['source_row_id']),
                    'split': split_name,
                    'metabolite_name': qrow['metabolite_name'],
                    'refmet_name': qrow['refmet_name'],
                    'metabolite_normalized': qrow['metabolite_normalized'],
                    'refmet_normalized': qrow['refmet_normalized'],
                    'gold_target_id': gold,
                    'embedding_rank': float(ranks_from_neighbor_ids(np.array([int(row_i)]), K_RETRIEVE)[0]),
                    'reranker_best_rank': None if not np.isfinite(best_rank) else int(best_rank),
                    'top1_refmet_name': target_display[top_target_id],
                    'top1_refmet_normalized': target_norms[top_target_id],
                    'top1_target_id': top_target_id,
                    'top1_correct': bool(top_target_id == gold),
                    'top1_embedding_score': float(emb_scores[local_i, top_j]),
                    'top1_rerank_score': top_score,
                    'top2_rerank_score': top2_score,
                    'confidence_margin': float(top_score - top2_score) if np.isfinite(top2_score) else float('nan'),
                })
                for rank0, j in enumerate(ordered[:int(SAVE_TOP_K)]):
                    target_id = int(cand_ids[local_i, int(j)])
                    top_rows.append({
                        'query_id': int(qrow['query_id']),
                        'rank': int(rank0 + 1),
                        'target_id': target_id,
                        'candidate_refmet_name': target_display[target_id],
                        'candidate_refmet_normalized': target_norms[target_id],
                        'correct': bool(target_id == gold),
                        'embedding_score': float(emb_scores[local_i, int(j)]),
                        'rerank_score': float(scores[local_i, int(j)]),
                    })
    ranks = np.concatenate(all_ranks) if all_ranks else np.array([], dtype=np.float64)
    metrics = metrics_from_ranks(ranks, prefix='rerank_')
    return metrics, pd.DataFrame(prediction_rows), pd.DataFrame(top_rows)


# Reranking is the expensive string-feature stage. We report validation and held-out test,
# while keeping the cheaper embedding baseline for train/val/test.
reranker_metrics = {}
val_metrics, _, _ = evaluate_reranker_split(split_indices['val'], 'val', save_predictions=False)
reranker_metrics['val'] = val_metrics
test_metrics, test_predictions, test_top_hits = evaluate_reranker_split(split_indices['test'], 'test', save_predictions=True)
reranker_metrics['test'] = test_metrics
print(json.dumps(reranker_metrics, indent=2))


## Confidence Threshold Analysis


In [ ]:
def confidence_curve(predictions: pd.DataFrame, coverage_levels=(1.0, 0.9, 0.75, 0.5, 0.25, 0.1)) -> pd.DataFrame:
    df = predictions.copy()
    df['confidence_margin'] = df['confidence_margin'].fillna(-1e9)
    df = df.sort_values('confidence_margin', ascending=False).reset_index(drop=True)
    rows = []
    n = len(df)
    for coverage in coverage_levels:
        keep_n = max(1, int(round(n * coverage)))
        kept = df.iloc[:keep_n]
        rows.append({
            'coverage': float(coverage),
            'n_answered': int(keep_n),
            'top1_accuracy_when_answered': float(kept['top1_correct'].mean()),
            'minimum_confidence_margin': float(kept['confidence_margin'].min()),
        })
    return pd.DataFrame(rows)


conf_curve = confidence_curve(test_predictions)
display(conf_curve)


## Save V3 Outputs


In [ ]:
combined_metrics = {
    'run_name': RUN_NAME,
    'model_key': 'sapbert_contrastive',
    'task': 'closed_set_lab_refmet_name_prediction_from_metabolite_name',
    'data_summary': load_json_if_exists(RESULTS_RUN_DIR / 'v3_closed_set_data_summary.json'),
    'embedding_baseline': baseline_metrics,
    'tuned_reranker': reranker_metrics,
    'best_reranker_weights': best_weights,
    'primary_test_result': {
        **baseline_metrics['test'],
        **reranker_metrics['test'],
    },
    'notes': [
        'Candidate labels are unique refmet_name values from the provided lab file.',
        'All lab-provided RefMet labels are searchable in train/val/test; query rows are held out by split.',
        'Reranker weights are selected using a validation subset and reported on the held-out test split.',
    ],
}
save_json(combined_metrics, RESULTS_RUN_DIR / 'v3_closed_set_metrics.json')
test_predictions.to_csv(RESULTS_RUN_DIR / 'v3_closed_set_test_predictions.csv', index=False)
test_top_hits.to_csv(RESULTS_RUN_DIR / 'v3_closed_set_test_top_hits.csv', index=False)
conf_curve.to_csv(RESULTS_RUN_DIR / 'v3_confidence_curve.csv', index=False)

comparison_rows = []
for split in ['train', 'val', 'test']:
    row = {'split': split}
    row.update(baseline_metrics[split])
    row.update(reranker_metrics.get(split, {}))
    comparison_rows.append(row)
comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(RESULTS_RUN_DIR / 'v3_closed_set_comparison.csv', index=False)

artifact_manifest = pd.DataFrame([
    {'field': 'config', 'value': str(RESULTS_RUN_DIR / 'config.json')},
    {'field': 'metrics', 'value': str(RESULTS_RUN_DIR / 'v3_closed_set_metrics.json')},
    {'field': 'comparison', 'value': str(RESULTS_RUN_DIR / 'v3_closed_set_comparison.csv')},
    {'field': 'test_predictions', 'value': str(RESULTS_RUN_DIR / 'v3_closed_set_test_predictions.csv')},
    {'field': 'test_top_hits', 'value': str(RESULTS_RUN_DIR / 'v3_closed_set_test_top_hits.csv')},
    {'field': 'confidence_curve', 'value': str(RESULTS_RUN_DIR / 'v3_confidence_curve.csv')},
    {'field': 'reranker_weights', 'value': str(RESULTS_RUN_DIR / 'v3_best_reranker_weights.json')},
    {'field': 'tuning_results', 'value': str(RESULTS_RUN_DIR / 'v3_reranker_tuning_results.csv')},
    {'field': 'refmet_dictionary', 'value': str(RESULTS_ARTIFACTS_DIR / 'v3_refmet_label_dictionary.csv')},
])
artifact_manifest.to_csv(RESULTS_RUN_DIR / 'v3_artifact_manifest.csv', index=False)
display(comparison)
display(artifact_manifest)


## Plots


In [ ]:
import matplotlib.pyplot as plt

plot_dir = RESULTS_RUN_DIR / 'plots'
plot_dir.mkdir(parents=True, exist_ok=True)

comparison = pd.read_csv(RESULTS_RUN_DIR / 'v3_closed_set_comparison.csv')
x = np.arange(len(comparison))

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - 0.18, comparison['embedding_MRR'], width=0.36, label='Embedding search')
ax.bar(x + 0.18, comparison['rerank_MRR'], width=0.36, label='Tuned hybrid reranker')
ax.set_xticks(x)
ax.set_xticklabels(comparison['split'])
ax.set_ylabel('MRR')
ax.set_title('V3 closed-set RefMet MRR')
ax.legend()
fig.tight_layout()
fig.savefig(plot_dir / 'v3_closed_set_mrr.png', dpi=180, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - 0.18, comparison['embedding_HitRate@1'], width=0.36, label='Embedding Hit@1')
ax.bar(x + 0.18, comparison['rerank_HitRate@1'], width=0.36, label='Reranker Hit@1')
ax.set_xticks(x)
ax.set_xticklabels(comparison['split'])
ax.set_ylim(0, 1)
ax.set_ylabel('Hit@1')
ax.set_title('V3 closed-set RefMet top-1 accuracy')
ax.legend()
fig.tight_layout()
fig.savefig(plot_dir / 'v3_closed_set_hit1.png', dpi=180, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(conf_curve['coverage'], conf_curve['top1_accuracy_when_answered'], marker='o')
ax.invert_xaxis()
ax.set_ylim(0, 1)
ax.set_xlabel('Coverage answered')
ax.set_ylabel('Top-1 accuracy')
ax.set_title('V3 confidence threshold curve on test split')
fig.tight_layout()
fig.savefig(plot_dir / 'v3_confidence_curve.png', dpi=180, bbox_inches='tight')
plt.show()

print('Saved plots to:', plot_dir)
